- format the code in the oop class

### [PROJECT-1] landing zone-ingest-transform-load by batch-processing [word count]

data movement->

base dir is project dir...

- ./database (permanent) -> (ingest) ./landing_dir -> transform -> (load) to delta table 

In [0]:
dbutils.fs.help()

dbutils.fs provides utilities for working with FileSystems. Most methods in
this package can take either a DBFS path (e.g., "/foo" or "dbfs:/foo"), or
another FileSystem URI.

For more info about a method, use dbutils.fs.help("methodName") .

In notebooks, you can also use the %fs shorthand to access DBFS. The %fs shorthand maps
straightforwardly onto dbutils calls. For example, "%fs head --maxBytes=10000 /file/path"
translates into "dbutils.fs.head("/file/path", maxBytes = 10000)".
 mount mount(source: String, mountPoint: String, encryptionType: String = "", owner: String = null, extraConfigs: Map = Map.empty[String, String]): boolean -> Mounts the given source directory into DBFS at the given mount point mounts: Seq -> Displays information about what is mounted within DBFS refreshMounts: boolean -> Forces all machines in this cluster to refresh their mount cache, ensuring they receive the most recent information unmount(mountPoint: String): boolean -> Deletes a DBFS mount point updateMount(source: String, mountPoint: String, encryptionType: String = "", owner: String = null, extraConfigs: Map = Map.empty[String, String]): boolean -> Similar to mount(), but updates an existing mount point (if present) instead of creating a new one fsutils cp(from: String, to: String, recurse: boolean = false): boolean -> Copies a file or directory, possibly across FileSystems head(file: String, maxBytes: int = 65536): String -> Returns up to the first 'maxBytes' bytes of the given file as a String encoded in UTF-8 ls(dir: String): Seq -> Lists the contents of a directory mkdirs(dir: String): boolean -> Creates the given directory if it does not exist, also creating any necessary parent directories mv(from: String, to: String, recurse: boolean = false): boolean -> Moves a file or directory, possibly across FileSystems put(file: String, contents: String, overwrite: boolean = false): boolean -> Writes the given String out to a file, encoded in UTF-8 rm(dir: String, recurse: boolean = false): boolean -> Removes a file or directory

In [0]:
from pyspark.sql.functions import explode, split, trim, lower

class WordCountProject:
    def __init__(self):
        #initialize the base dir
        self.project_file_dir="dbfs:/FileStore/project1/"
        self.dataset_dir="dataset/"
        self.landing_zone="landing_zone/"
        self.load_table_name="wordCountTable"   
        self.check_point_dir="check_point"
        self.resul_df=None
    def cleanup_n_setup(self):
        # 1. cleanup code
        #drop the pre-existing load table
        drop_query=f"drop table if exists {self.load_table_name}"
        spark.sql(drop_query)
        #delete the dirs where the datafile have been stored for the files 
        dbutils.fs.rm(f"/user/hive/warehouse/{self.load_table_name}",True)

        #delete the lending zone and check-pointing dir
        dbutils.fs.rm(self.project_file_dir+self.landing_zone,True)
        dbutils.fs.rm(self.project_file_dir+self.check_point_dir,True)
       
        # 2. setup code

        dbutils.fs.mkdirs(self.project_file_dir+self.landing_zone)
        dbutils.fs.mkdirs(self.project_file_dir+self.check_point_dir)
        print("CLEANUP & SETUP RES: data cleanup and setup successfull !")

    def ingest_data(self):
        # copy all the text data files from dataset dir to the landing zone
        res=dbutils.fs.cp(self.project_file_dir+self.dataset_dir, self.project_file_dir+self.landing_zone, True)
        res_str="all text files are copied from the dataset dir to the landing zone" if res else "Copying process failed !"
        print("INGESTION RES:",res_str)


    def extract_data_from_landing_zone(self):
        # read the Data from the landing zone
        df=spark.read.format("text").option("lineSep",".").load(self.project_file_dir+self.landing_zone+"*.txt")
        # display(df.head(10))
        print("EXTRACTION RES: data extraction successfull")
        return df
    

    def transform_data(self, df):
        word_df=df.select(explode(split(df.value," ")).alias("word"))
        quality_df=word_df.select(trim(lower(word_df.word)).alias("word"))
        quality_df=quality_df.where("word is not null").where("word rlike '[a-z]'")
        word_count_df=quality_df.groupBy("word").count()
        # display(word_count_df.head(10))
        print("TRANSFOMATION RES: data transformation successfull !")
        return word_count_df

    def load_data(self,df):
        df.write.format("delta").mode("overwrite").saveAsTable(self.load_table_name)
        print("LOAD RES: data load to delta-table successfully ! \nTABLE NAME:", self.load_table_name)

    def process_data(self):
        # cleanup & setup
        print("-"*10, " HOUSEKEEPING ROCESS STARTED ", "-"*10,)
        self.cleanup_n_setup()

        # ingest tha data to the landing zone
        print("-"*10, " INGESTION PROCESS STARTED ", "-"*10,)
        self.ingest_data()


        ## ETL
        print("-"*10, " ETL PROCESS STARTED ", "-"*10,)

        # Extract data 
        extracted_data_df=self.extract_data_from_landing_zone()

        # transform data
        transformed_df=self.transform_data(extracted_data_df)
        self.resul_df=transformed_df

        # Load data to sink
        self.load_data(transformed_df)

    def get_result_df(self):
        return self.resul_df




In [0]:
# wc_proj=WordCountProject()
# wc_proj.process_data()

----------  HOUSEKEEPING ROCESS STARTED  ----------
CLEANUP & SETUP RES: data cleanup and setup successfull !
----------  INGESTION PROCESS STARTED  ----------
INGESTION RES: all text files are copied from the dataset dir to the landing zone
----------  ETL PROCESS STARTED  ----------
EXTRACTION RES: data extraction successfull
TRANSFOMATION RES: data transformation successfull !
LOAD RES: data load to delta-table successfully ! 
TABLE NAME: wordCountTable


In [0]:
# display(wc_proj.get_result_df())

word,count
ensures,1
stream,2
fault-tolerance,2
will,2
you,5
can,3
"java,",1
guarantees,3
arrive,1
system,1


Databricks visualization. Run in Databricks to view.

### Project Completed !